In [5]:
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader

from transformers import BertTokenizer, BertModel

from sklearn.model_selection import train_test_split
from sklearn.metrics import accuracy_score

import pandas as pd
import numpy as np
from tqdm import tqdm

In [6]:
RAW_DATA = [
    {
        "job_description": "Software engineer with Python, Django, and REST API experience.",
        "resume": "3 years of Python development. Built REST APIs with Django and Flask.",
        "label": 1,
    },
    {
        "job_description": "Data scientist skilled in machine learning, pandas, and scikit-learn.",
        "resume": "Data scientist with 4 years experience in ML, pandas, and scikit-learn pipelines.",
        "label": 1,
    },
    {
        "job_description": "Frontend developer with React, JavaScript, and CSS expertise.",
        "resume": "Built responsive web apps using React and modern JavaScript (ES6+). Strong CSS skills.",
        "label": 1,
    },
    {
        "job_description": "DevOps engineer experienced with Docker, Kubernetes, and CI/CD pipelines.",
        "resume": "Managed cloud infrastructure using Docker containers and Kubernetes. Set up GitHub Actions CI/CD.",
        "label": 1,
    },
    {
        "job_description": "NLP engineer with Hugging Face Transformers and BERT fine-tuning experience.",
        "resume": "Fine-tuned BERT and RoBERTa models for text classification using Hugging Face Transformers.",
        "label": 1,
    },
    {
        "job_description": "Backend engineer proficient in Java, Spring Boot, and PostgreSQL.",
        "resume": "5 years building microservices with Java Spring Boot. PostgreSQL database design.",
        "label": 1,
    },
    {
        "job_description": "Mobile developer with iOS, Swift, and Xcode experience.",
        "resume": "Published 3 iOS apps using Swift and Xcode. Familiar with UIKit and SwiftUI.",
        "label": 1,
    },
    {
        "job_description": "Cybersecurity analyst with penetration testing and network security skills.",
        "resume": "Certified ethical hacker. Conducted pen tests and vulnerability assessments on enterprise networks.",
        "label": 1,
    },
    {
        "job_description": "Software engineer with Python, Django, and REST API experience.",
        "resume": "Marketing manager with 5 years in brand strategy and social media campaigns.",
        "label": 0,
    },
    {
        "job_description": "Data scientist skilled in machine learning, pandas, and scikit-learn.",
        "resume": "Registered nurse with experience in ICU patient care and medication administration.",
        "label": 0,
    },
    {
        "job_description": "Frontend developer with React, JavaScript, and CSS expertise.",
        "resume": "Chartered accountant specialising in tax returns and financial audits.",
        "label": 0,
    },
    {
        "job_description": "DevOps engineer experienced with Docker, Kubernetes, and CI/CD pipelines.",
        "resume": "Primary school teacher with a passion for literacy and classroom management.",
        "label": 0,
    },
    {
        "job_description": "NLP engineer with Hugging Face Transformers and BERT fine-tuning experience.",
        "resume": "Graphic designer skilled in Adobe Illustrator, Photoshop, and brand identity.",
        "label": 0,
    },
    {
        "job_description": "Backend engineer proficient in Java, Spring Boot, and PostgreSQL.",
        "resume": "Chef with 8 years experience in French cuisine and restaurant kitchen management.",
        "label": 0,
    },
    {
        "job_description": "Mobile developer with iOS, Swift, and Xcode experience.",
        "resume": "Civil engineer specialising in structural analysis and construction project management.",
        "label": 0,
    },
    {
        "job_description": "Cybersecurity analyst with penetration testing and network security skills.",
        "resume": "Human resources manager focusing on recruitment, payroll, and employee relations.",
        "label": 0,
    },
]

# Positive match = 1
# Negative match = 0

In [7]:
TOKENIZER = BertTokenizer.from_pretrained("bert-base-uncased")

sample_text = "Python developer with Django experience."
tokens = TOKENIZER(sample_text, return_tensors="pt")

MAX_LENGTH = 64

In [8]:
class JobResumeDataset(Dataset):

    def __init__(self, data):
        self.data = data

    def __len__(self):
        return len(self.data)

    def __getitem__(self, index):
        pair   = self.data[index]
        jd     = pair["job_description"]
        resume = pair["resume"]
        label  = pair["label"]

        jd_tokens = TOKENIZER(
            jd,
            max_length=MAX_LENGTH,
            padding="max_length",
            truncation=True,
            return_tensors="pt",
        )

        resume_tokens = TOKENIZER(
            resume,
            max_length=MAX_LENGTH,
            padding="max_length",
            truncation=True,
            return_tensors="pt",
        )

        return {
            "jd_ids":          jd_tokens["input_ids"].squeeze(0),
            "jd_mask":         jd_tokens["attention_mask"].squeeze(0),
            "resume_ids":      resume_tokens["input_ids"].squeeze(0),
            "resume_mask":     resume_tokens["attention_mask"].squeeze(0),
            "label":           torch.tensor(label, dtype=torch.float),
        }

In [9]:
train_data, test_data = train_test_split(RAW_DATA, test_size=0.2, random_state=42)

train_dataset = JobResumeDataset(train_data)
test_dataset  = JobResumeDataset(test_data)

train_loader = DataLoader(train_dataset, batch_size=4, shuffle=True)
test_loader  = DataLoader(test_dataset,  batch_size=4, shuffle=False)

In [10]:
class SiameseBERT(nn.Module):

    def __init__(self):
        super().__init__()
        self.bert = BertModel.from_pretrained("bert-base-uncased")
        self.fc = nn.Linear(768, 128)

    def encode_text(self, input_ids, attention_mask):
        bert_output = self.bert(input_ids=input_ids, attention_mask=attention_mask)
        token_embeddings = bert_output.last_hidden_state
        mask = attention_mask.unsqueeze(-1).float()
        sum_embeddings  = (token_embeddings * mask).sum(dim=1)
        num_real_tokens = mask.sum(dim=1)
        mean_embedding  = sum_embeddings / num_real_tokens
        embedding = self.fc(mean_embedding)
        embedding = F.normalize(embedding, p=2, dim=1)
        return embedding

    def forward(self, jd_ids, jd_mask, resume_ids, resume_mask):
        jd_embedding     = self.encode_text(jd_ids,     jd_mask)
        resume_embedding = self.encode_text(resume_ids, resume_mask)
        similarity = (jd_embedding * resume_embedding).sum(dim=1)
        return similarity, jd_embedding, resume_embedding

In [11]:
class ContrastiveLoss(nn.Module):

    def __init__(self, margin=1.0):
        super().__init__()
        self.margin = margin

    def forward(self, embedding_a, embedding_b, labels):
        distance = F.pairwise_distance(embedding_a, embedding_b)
        positive_loss = labels * distance.pow(2)
        negative_loss = (1 - labels) * F.relu(self.margin - distance).pow(2)
        total_loss = (positive_loss + negative_loss).mean()
        return total_loss

In [12]:
DEVICE     = torch.device("cuda" if torch.cuda.is_available() else "cpu")
EPOCHS     = 3
LR         = 2e-5

model     = SiameseBERT().to(DEVICE)
criterion = ContrastiveLoss(margin=1.0)

optimiser = torch.optim.AdamW(model.parameters(), lr=LR)

In [13]:


for epoch in range(1, EPOCHS + 1):

    model.train()
    total_loss = 0.0

    for batch in tqdm(train_loader, desc=f"Epoch {epoch}/{EPOCHS} [Train]"):
        jd_ids     = batch["jd_ids"].to(DEVICE)
        jd_mask    = batch["jd_mask"].to(DEVICE)
        res_ids    = batch["resume_ids"].to(DEVICE)
        res_mask   = batch["resume_mask"].to(DEVICE)
        labels     = batch["label"].to(DEVICE)

        optimiser.zero_grad()
        similarity, emb_jd, emb_resume = model(jd_ids, jd_mask, res_ids, res_mask)
        loss = criterion(emb_jd, emb_resume, labels)
        loss.backward()
        optimiser.step()
        total_loss += loss.item()

    avg_train_loss = total_loss / len(train_loader)

    model.eval()
    all_predictions = []
    all_true_labels = []

    with torch.no_grad():
        for batch in test_loader:
            jd_ids   = batch["jd_ids"].to(DEVICE)
            jd_mask  = batch["jd_mask"].to(DEVICE)
            res_ids  = batch["resume_ids"].to(DEVICE)
            res_mask = batch["resume_mask"].to(DEVICE)
            labels   = batch["label"].to(DEVICE)

            similarity, _, _ = model(jd_ids, jd_mask, res_ids, res_mask)
            predictions = (similarity > 0.5).long()
            all_predictions.extend(predictions.cpu().numpy())
            all_true_labels.extend(labels.cpu().long().numpy())

    accuracy = accuracy_score(all_true_labels, all_predictions)
    print(f"\n Epoch {epoch} Results:")
    print(f"Train Loss : {avg_train_loss:.4f}")
    print(f"Test Acc   : {accuracy * 100:.1f}%\n")

print("Training complete \n")

Epoch 1/3 [Train]: 100%|██████████| 3/3 [00:06<00:00,  2.22s/it]



 Epoch 1 Results:
Train Loss : 0.1576
Test Acc   : 75.0%



Epoch 2/3 [Train]: 100%|██████████| 3/3 [00:04<00:00,  1.57s/it]



 Epoch 2 Results:
Train Loss : 0.0740
Test Acc   : 75.0%



Epoch 3/3 [Train]: 100%|██████████| 3/3 [00:06<00:00,  2.24s/it]



 Epoch 3 Results:
Train Loss : 0.0457
Test Acc   : 75.0%

Training complete 



In [14]:
def score_match(job_description: str, resume: str) -> float:
    model.eval()
 
    jd_tokens = TOKENIZER(
        job_description, max_length=MAX_LENGTH,
        padding="max_length", truncation=True, return_tensors="pt"
    )
    res_tokens = TOKENIZER(
        resume, max_length=MAX_LENGTH,
        padding="max_length", truncation=True, return_tensors="pt"
    )
 
    jd_ids   = jd_tokens["input_ids"].to(DEVICE)
    jd_mask  = jd_tokens["attention_mask"].to(DEVICE)
    res_ids  = res_tokens["input_ids"].to(DEVICE)
    res_mask = res_tokens["attention_mask"].to(DEVICE)
 
    with torch.no_grad():
        similarity, _, _ = model(jd_ids, jd_mask, res_ids, res_mask)
 
    score = (similarity.item() + 1) / 2
    return round(score, 3)

In [15]:
job1 = "Looking for a Python developer with machine learning and NLP experience."
job2= "Looking for a java developer for backed development."
 
candidates = [
    {
        "name"  : "Alice",
        "resume": "Python developer with 4 years in NLP and machine learning. Used BERT and spaCy.",
    },
    {
        "name"  : "Bob",
        "resume": "Java backend developer. Built REST APIs and microservices. No ML experience.",
    },
    {
        "name"  : "Carol",
        "resume": "Data analyst using Python, pandas, and basic scikit-learn models.",
    },
    {
        "name"  : "Dan",
        "resume": "Graphic designer. Skilled in Photoshop, Illustrator, and brand design.",
    },
]
 
print(f"\nJob Description:\n   {job1}\n")
print(f"{'Candidate':<10} {'Score':>8}   {'Match Strength'}")
print("-" * 50)
 
results1 = []
results2 = []
for candidate in candidates:
    score1 = score_match(job1, candidate["resume"])
    score2 = score_match(job2, candidate["resume"])
    results1.append((candidate["name"], score1))
    results2.append((candidate["name"], score2))

results1.sort(key=lambda x: x[1], reverse=True)
results2.sort(key=lambda x: x[1], reverse=True)


for name, score in results1:
    bar_length = int(score * 30)
    bar = "█" * bar_length + "░" * (30 - bar_length)
    print(f"{name:<10} {score:>8.3f}   {bar}")

print()
print(f"Best candidate: {results1[0][0]} (score = {results1[0][1]})")


print(f"\nJob Description:\n   {job2}\n")
print(f"{'Candidate':<10} {'Score':>8}   {'Match Strength'}")
print("-" * 50)

for name, score in results2:
    bar_length = int(score * 30)
    bar = "█" * bar_length + "░" * (30 - bar_length)
    print(f"{name:<10} {score:>8.3f}   {bar}")
 
print()
print(f"Best candidate: {results2[0][0]} (score = {results2[0][1]})")


Job Description:
   Looking for a Python developer with machine learning and NLP experience.

Candidate     Score   Match Strength
--------------------------------------------------
Alice         0.988   █████████████████████████████░
Carol         0.975   █████████████████████████████░
Bob           0.974   █████████████████████████████░
Dan           0.851   █████████████████████████░░░░░

Best candidate: Alice (score = 0.988)

Job Description:
   Looking for a java developer for backed development.

Candidate     Score   Match Strength
--------------------------------------------------
Alice         0.963   ████████████████████████████░░
Bob           0.951   ████████████████████████████░░
Carol         0.929   ███████████████████████████░░░
Dan           0.835   █████████████████████████░░░░░

Best candidate: Alice (score = 0.963)


In [16]:
import os

SAVE_DIR = "/Users/ananya/development/optimal_hire/model/siamese_bert_model"
os.makedirs(SAVE_DIR, exist_ok=True)

torch.save(model.state_dict(), os.path.join(SAVE_DIR, "siamese_bert_weights.pt"))

model.bert.save_pretrained(os.path.join(SAVE_DIR, "bert"))
TOKENIZER.save_pretrained(os.path.join(SAVE_DIR, "bert"))

import json
config = {
    "max_length": MAX_LENGTH,
    "fc_in_features": 768,
    "fc_out_features": 128,
}
with open(os.path.join(SAVE_DIR, "config.json"), "w") as f:
    json.dump(config, f)

print(f"\nModel saved to '{SAVE_DIR}/' directory")
print("   Files created:")
print("     • siamese_bert_weights.pt  (full model weights)")
print("     • bert/                    (BERT model + tokenizer)")
print("     • config.json              (metadata)")


Model saved to '/Users/ananya/development/optimal_hire/model/siamese_bert_model/' directory
   Files created:
     • siamese_bert_weights.pt  (full model weights)
     • bert/                    (BERT model + tokenizer)
     • config.json              (metadata)
